# Итоги для кейса VK Case Champ

Синтетические данные за 30 дней сентября 2026 года. Цели кейса — больше бизнесов используют инструменты продаж ВК и больше заказов оформляются внутри ВК. Ниже — опорные выводы из агрегированных таблиц, без анализа `Business_groups.csv` и рекламных расходов.

## Как читать маршруты: уточнение для анализа

Ниже используется **рабочая трактовка, уточнённая командой**, а не доказанное по выгрузке техническое определение каждого события:

- **Маркет VK:** открытие карточки товара, относящегося к Маркету, во VK → нажатие кнопки → возможный переход к чекауту VK или в чат с продавцом. Кейс упоминает и внешние сайты; фактические назначения кнопок в выгрузке не разделены.
- **Партнёр 1 через карточку VK:** открытие карточки его товара во VK → нажатие кнопки → переход к чекауту на сайте партнёра (по уточнённому сценарию). Это ещё не оформленный заказ.
- **Партнёр 1 напрямую:** переход из VK без открытия карточки во VK → открытие страницы товара на сайте партнёра (по уточнённому сценарию); в данных этот маршрут отражён как прямые переходы, а не как открытия карточки VK.

Партнёр 1 — условная площадка, не обязательно Ozon. Товар Маркета может продаваться и на других площадках. Назначения кнопок и факт открытия страницы партнёра нужно подтвердить по справочнику событий и интеграции; эти детали отдельно в CSV не измерены. Прямые переходы и нажатия ведут на разные шаги и могут пересекаться по пользователям: их сумма не является числом уникальных покупателей или единым знаменателем конверсии. Сравнение агрегированных отношений — не когортная конверсия и не диагностика причин.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path.cwd().parent / 'Data'
mv = 'Маркет Вконтакте'
p1 = 'Партнёр 1'
pd.set_option('display.float_format', '{:,.2f}'.format)

market = pd.read_csv(data_path / 'Market_Partner.csv').set_index('Метрика')
traffic = pd.read_csv(data_path / 'Market_Partner_traffic.csv').set_index('Источник')
total = pd.read_csv(data_path / 'Market_Total.csv').set_index('Период')
category = pd.read_csv(data_path / 'Market_Partner_categories.csv').set_index('Категория')
kpi = market.loc[['Показы товаров', 'Открытия карточки товара', 'Нажатия на кнопку в карточке', 'Оформленные заказы', 'GMV, ₽', 'Средний чек, ₽', 'Уникальные покупатели за 30 дней', 'Заказов на покупателя', 'Доля повторных покупателей (2+ заказа), %'], [mv, p1]].copy()
kpi['Доля Маркета в сумме каналов, %'] = np.nan
for metric in ['Показы товаров', 'Открытия карточки товара', 'Нажатия на кнопку в карточке', 'Оформленные заказы', 'GMV, ₽', 'Уникальные покупатели за 30 дней']:
    kpi.loc[metric, 'Доля Маркета в сумме каналов, %'] = kpi.loc[metric, mv] / (kpi.loc[metric, mv] + kpi.loc[metric, p1]) * 100
kpi

,Маркет Вконтакте,Партнёр 1,"Доля Маркета в сумме каналов, %"
Метрика,,,
Показы товаров,"4,777,750,023.00","5,729,628,790.00",45.47
Открытия карточки товара,"193,512,461.00","86,882,822.00",69.01
Нажатия на кнопку в карточке,"5,969,063.00","36,626,075.00",14.01
Оформленные заказы,"413,740.00","4,927,525.00",7.75
"GMV, ₽","1,564,546,740.00","5,119,514,685.00",23.41
"Средний чек, ₽","3,781.00","1,039.00",NaN
Уникальные покупатели за 30 дней,"296,148.00","2,901,334.00",9.26
Заказов на покупателя,1.40,1.70,NaN
"Доля повторных покупателей (2+ заказа), %",19.00,26.60,NaN


In [ ]:
# Интерактивные визуализации
import plotly.graph_objects as go

def show_plotly(fig, title, x_title, y_title, height=500):
    fig.update_layout(
        title=dict(text=title), template='plotly_white', height=height,
        xaxis_title=x_title, yaxis_title=y_title,
        legend_title_text='Канал', margin=dict(l=70, r=30, t=80, b=90),
        font=dict(size=13), hoverlabel=dict(namelength=-1),
    )
    fig.show(config={'displaylogo': False, 'responsive': True})


### Интерактивно: вклад Маркета в показатели двух каналов
Доли рассчитаны относительно суммы двух каналов; аудитории могут пересекаться. Это не доля всего рынка.


In [ ]:
share_metrics = ['Показы товаров', 'Открытия карточки товара', 'Нажатия на кнопку в карточке',
                 'Оформленные заказы', 'GMV, ₽', 'Уникальные покупатели за 30 дней']
share = kpi.loc[share_metrics, 'Доля Маркета в сумме каналов, %']
fig = go.Figure(go.Bar(x=share.iloc[::-1], y=share.index[::-1], orientation='h', marker_color='#3978C6',
                       hovertemplate='%{y}<br>Доля Маркета: %{x:.2f}%<extra></extra>'))
fig.update_xaxes(range=[0, 100], ticksuffix='%')
show_plotly(fig, 'Масштаб Маркета в сумме двух каналов', 'Доля Маркета, %', '', 500)


Интерактивный график Plotly: Масштаб Маркета в сумме двух каналов

## 1. Место ВК среди площадок

Маркет привлекает много открытий, но партнёр значительно превосходит его по числу заказов; у Маркета существенно выше средний чек. Это сравнение только с одним партнёром, а не оценка доли ВК среди всех маркетплейсов и классифайдов. Заказы партнёра могут включать прямые переходы без открытия карточки ВК, а покупки за пределами ВК видны не полностью.
Партнёрский прямой маршрут ведёт на страницу товара партнёра без открытия карточки VK; нажатие в карточке VK, согласно уточнению команды, ведёт к чекауту партнёра. Доля нажатий в карточке Маркета не сопоставима с партнёрской как конверсия: следующие шаги различны.


## 2. Для кого и какие боли проверять

Продукты питания и хобби — заметные категории по заказам Маркета. Гипотеза: начать с продавцов, для которых важны сообщество, контент и повторное взаимодействие, а также проверить сценарии локальных продаж. Категория не доказывает ни размер бизнеса, ни его потребности: сегментацию следует проверить интервью; таблица бизнес-сообществ исключена из анализа.

Отношение нажатий к открытиям и доля учтённых оплат среди оформленных — сигналы для проверки, но не диагноз проблем карточки или чекаута. Кнопки ведут к разным действиям, а внешние оплаты могут не наблюдаться; сначала проверить назначение кнопок и учёт событий.

## 3. УТП и инструменты

Гипотеза УТП: продавать там, где бизнес уже общается с аудиторией, соединяя контент и сообщество с каталогом и простым оформлением заказа. Проверить улучшения карточек, публикаций товаров в ленте и клипах, поиска, чекаута и инструментов повторных продаж. Ключевые метрики экспериментов: активные бизнесы с товарами и продажами, оформленные заказы внутри ВК, а также качество заказов и оплаты. Разделять результат ВК и партнёра; измерять перенос, а не предполагать каннибализацию.

## 4. Роль Маркета

Главная отдельного Маркета даёт небольшую долю открытий, тогда как витрины магазинов и другие поверхности значимы. Предварительно Маркет разумно рассматривать как инфраструктуру каталога, поиска и оформления на разных поверхностях ВК, а не делать единственную ставку на самостоятельную витрину. Это гипотеза о роли продукта, а не доказательство, что отдельная витрина не нужна: «Другое» крупное, данные об исходах по источникам отсутствуют.

In [2]:
sources = traffic.loc[~traffic.index.str.startswith('в т.ч.')]
role = pd.DataFrame({'Открытия Маркета': sources[mv], 'Доля открытий Маркета, %': sources[mv] / sources[mv].sum() * 100, 'Открытия партнёра': sources[p1], 'Доля открытий партнёра, %': sources[p1] / sources[p1].sum() * 100})
role

,Открытия Маркета,"Доля открытий Маркета, %",Открытия партнёра,"Доля открытий партнёра, %"
Источник,,,,
Другое,"77,753,037.00",40.18,5380616,6.19
Витрина магазина,"75,604,323.00",39.07,714540,0.82
Лента,"18,795,696.00",9.71,15622386,17.98
Клипы,"17,239,066.00",8.91,64917109,74.72
Поиск,"3,707,521.00",1.92,159788,0.18
Главная Маркета Вконтакте,"412,818.00",0.21,88383,0.10


### Интерактивно: откуда открывают карточки Маркета
Доли относятся к сумме учтённых источников открытий Маркета. «Другое» сохранено; данных о заказах по источникам нет.


In [ ]:
role_plot = role.sort_values('Доля открытий Маркета, %')
fig = go.Figure(go.Bar(x=role_plot['Доля открытий Маркета, %'], y=role_plot.index, orientation='h',
                       marker_color='#3978C6', customdata=role_plot['Открытия Маркета'],
                       hovertemplate='%{y}<br>Доля: %{x:.2f}%<br>Открытий: %{customdata:,.0f}<extra></extra>'))
fig.update_xaxes(rangemode='tozero', ticksuffix='%')
show_plotly(fig, 'Источники открытий карточек Маркета', 'Доля в сумме источников Маркета, %', '', 500)


Интерактивный график Plotly: Источники открытий карточек Маркета

## 5. Путь бизнеса и проверка гипотез

1. Подключить сообщество и добавить товары: измерить долю бизнесов, завершивших настройку и опубликовавших доступный каталог.
2. Получить качественные показы через сообщество, поиск и контент: проверить видимость и переходы, не складывая пересекающиеся источники.
3. Разделить маршруты Маркета: карточка → чат, чекаут VK или иной переход; исследовать назначения кнопок, условия заказа и оплату на уровне когорт.
4. Вернуть покупателя: тестировать повторные коммуникации и частоту повторных заказов с учётом согласий и качества сервиса.
5. Для партнёра отдельно учитывать путь через карточку VK к чекауту партнёра и прямой переход на страницу товара; проверять статусы и атрибуцию. Экономику и ROMI без сопоставимых затрат и доходов не рассчитывать.

## Ограничения и следующие шаги

- Синтетические данные и один месяц не подтверждают причинность, сезонность или репрезентативность рынка.
- Разные назначения кнопок, прямые переходы партнёра на страницу товара и внешние оплаты нарушают сопоставимость простой воронки. Маршруты команды требуют сверки со справочником событий.
- Суммы заказов по категориям и в разбивках не равны общим итогам; классификацию и покрытие нужно уточнить. «Другое» существенно, а данные по заказам по источникам и GMV по категориям отсутствуют.
- Дневные уникальные пользователи не складываются в месячных уникальных. У партнёра велика доля покупателей без указанного региона.
- Расходы на рекламу в этих семи CSV не анализировались; таблица бизнес-сообществ исключена. Эффективность рекламы и ROMI оценить нельзя.